# RL Locomotion - Training on Google Colab

PPO training of locomotion policies. This notebook runs the local `src/train.py` on a free Colab T4 GPU.

## Configuration

Edit the variables in the next cell, then `Runtime -> Run all`.

In [ ]:
# === EDIT ME ===
GITHUB_REPO   = "https://github.com/<your-username>/<your-repo>.git"   # TODO: set your repo URL
GITHUB_BRANCH = "main"                                                 # or master
GITHUB_TOKEN  = ""                                                     # leave empty if public

RUN_NAME    = "baseline_ant"
CONFIG      = "configs/baseline_ant.yaml"
ENV_ID      = "Ant-v4"
TIMESTEPS   = 2_500_000
DEVICE      = "cuda"

DRIVE_RUNS  = "/content/drive/MyDrive/rl-locomotion/runs"
REPO_DIR    = "/content/rl-locomotion"
SAVE_DIR    = f"{DRIVE_RUNS}/{RUN_NAME}"
print(f"Save dir: {SAVE_DIR}")

## 1. Mount Drive, clone repo, install deps

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, subprocess, pathlib
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.chdir("/content")

if not pathlib.Path(REPO_DIR).exists():
    url = GITHUB_REPO
    if GITHUB_TOKEN:
        url = url.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--branch", GITHUB_BRANCH, url, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

os.chdir(REPO_DIR)
print("Repo at:", os.getcwd())
print(subprocess.check_output(["git", "log", "--oneline", "-3"]).decode())

In [ ]:
# install PyTorch (CUDA) + project deps
subprocess.check_call(["pip", "install", "-q",
    "torch==2.7.0", "--index-url", "https://download.pytorch.org/whl/cu118"])
subprocess.check_call(["pip", "install", "-q", "-r", "requirements.txt"])
print("deps installed")

## 2. Verify GPU

In [ ]:
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Run training

Calls `src/train.py` and streams logs into the notebook. Final model and TensorBoard events are saved to Drive.

In [ ]:
import sys, subprocess
sys.path.insert(0, REPO_DIR)
from src.envs.path_utils import patch_gym_make
patch_gym_make()

cmd = [
    "python", "src/train.py",
    "--config",   CONFIG,
    "--env",      ENV_ID,
    "--timesteps", str(TIMESTEPS),
    "--run-name", RUN_NAME,
    "--save-dir", SAVE_DIR,
    "--device",   DEVICE,
]
print(" ", " ".join(cmd))
proc = subprocess.run(cmd, check=False)
print(f"exit code: {proc.returncode}")

## 4. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{DRIVE_RUNS}" --port 6006

## 5. Download artefacts to local

After the cell above finishes, go to `MyDrive/rl-locomotion/runs/<RUN_NAME>/` and download:

- `model.zip`         -> local `experiments/models/<RUN_NAME>/model.zip`
- `monitor.csv`       -> local `experiments/logs/<RUN_NAME>/monitor.csv`
- `tb/` (optional)    -> local `experiments/logs/<RUN_NAME>/tb/`

Then continue with Sprint 1 analysis on local.